# Thrifted Jeans causal strategy audit

This notebook executes the self-contained research module in this directory. It reads only the supplied Round 1 Jeans CSV and never imports or changes the production algorithm or simulator. The original hybrid is replayed with its original stateful entry/holding/reversal mechanics; the corrected hybrid uses an EMA available only through the previous decision day.

In [1]:
from pathlib import Path
import sys
import pandas as pd

AUDIT_DIR = Path.cwd() / "research" / "thrifted_jeans"
if not (AUDIT_DIR / "jeans_audit.py").exists():
    AUDIT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research" / "thrifted_jeans" / "jeans_audit.py").exists()) / "research" / "thrifted_jeans"
sys.path.insert(0, str(AUDIT_DIR))
from jeans_audit import run_audit

print(AUDIT_DIR)

D:\Documents\Algojam\research\thrifted_jeans


## Execute the reproducible audit

The default run uses 500 reconstructed price paths for each moving-block length 5, 10, and 20, and 500 complete-family permutation paths. These are Monte Carlo diagnostics, not claims about unseen structural regimes.

In [2]:
result = run_audit(
    output_dir=AUDIT_DIR / "outputs",
    figure_dir=AUDIT_DIR / "figures",
    bootstrap_repetitions=500,
    familywise_repetitions=500,
)
result["comparison"][["P&L", "Incremental vs always long", "Max drawdown", "Turnover"]].round(2)

Thrifted Jeans audit complete
                                              P&L  Incremental vs always long  Max drawdown  Turnover
Candidate                                                                                            
always_long                               38136.0                         0.0      -44856.0       800
flat                                          0.0                    -38136.0           0.0         0
simple_kalman_K2                         100016.0                     61880.0      -26672.0     29600
kalman_trend_only_K2                      49920.0                     11784.0      -64152.0     52000
original_ema_mean_reversion                6848.0                    -31288.0      -64024.0     50400
corrected_ema_mean_reversion_prevstd      -3488.0                    -41624.0      -67832.0     45600
corrected_ema_mean_reversion_throughstd    -928.0                    -39064.0      -67832.0     47200
original_hybrid                          162496.0   

D:\Documents\Algojam\research\thrifted_jeans\jeans_audit.py:1346: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  im = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="coolwarm")
D:\Documents\Algojam\research\thrifted_jeans\jeans_audit.py:1346: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  im = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="coolwarm")


,P&L,Incremental vs always long,Max drawdown,Turnover
Candidate,,,,
always_long,38136.0,0.0,-44856.0,800
flat,0.0,-38136.0,0.0,0
simple_kalman_K2,100016.0,61880.0,-26672.0,29600
kalman_trend_only_K2,49920.0,11784.0,-64152.0,52000
original_ema_mean_reversion,6848.0,-31288.0,-64024.0,50400
corrected_ema_mean_reversion_prevstd,-3488.0,-41624.0,-67832.0,45600
corrected_ema_mean_reversion_throughstd,-928.0,-39064.0,-67832.0,47200
original_hybrid,162496.0,124360.0,-14536.0,58400
corrected_hybrid_prevstd,151584.0,113448.0,-14536.0,53600


In [3]:
checks = result["correctness"]
assert bool(checks["Value"].all())
assert result["manifest"]["production_files_modified"] is False
assert result["manifest"]["reference_pnls"]["always_long"] == 38136.0
assert result["manifest"]["reference_pnls"]["simple_kalman_K2"] == 100016.0
assert result["manifest"]["reference_pnls"]["original_hybrid"] == 162496.0
print("All causal, timing, integer-position, limit, P&L-convention, and reproduction checks passed.")
print("Requested CSV outputs:")
print(sorted(p.name for p in (AUDIT_DIR / "outputs").glob("*.csv")))
print("Requested figures:")
print(sorted(p.name for p in (AUDIT_DIR / "figures").glob("*.png")))

All causal, timing, integer-position, limit, P&L-convention, and reproduction checks passed.
Requested CSV outputs:
['candidate_comparison.csv', 'chronological_splits.csv', 'concentration_diagnostics.csv', 'correctness_checks.csv', 'ema_predictive_regressions.csv', 'familywise_null.csv', 'fixed_pnl_bootstrap_diagnostic.csv', 'parameter_sensitivity.csv', 'parameter_sensitivity_summary.csv', 'price_path_bootstrap.csv', 'regime_attribution.csv', 'slope_bucket_returns.csv', 'stress_placebo_tests.csv']
Requested figures:
['cumulative_pnl_comparison.png', 'hybrid_alpha_reversion_heatmap.png', 'kalman_q_sensitivity_heatmap.png', 'next_day_returns_by_slope_bucket.png', 'pnl_by_chronological_segment.png', 'positions_against_price.png']


The associated JEANS_STRATEGY_AUDIT.md report is generated after this notebook run from the saved CSV outputs. The fixed-realised-P&L bootstrap is retained only as a labelled diagnostic; the primary uncertainty table is the price-path bootstrap that reconstructs prices and reruns every path-dependent strategy.